#  Baseline Feasibility Test for Change Prediction

Answer one prerequisite question: can the t-period balance sheet predict the t to t+1 financial change? This notebook only tests whether signal exists and does not train the final model and does not tune hyperparameters.

**Data**：`03_financial_change_labels.csv`, 81,603 company-period pair。

**Decision rule**

| Outcome | Interpretation | Next step |
|---|---|---|
| skill ≤ 0 & Spearman < 0.05 | no usable signal | -- |
| skill ≈ 0 but Spearman 0.05–0.15 | ranking-only signal | -- |
| skill > 0.05 or Spearman > 0.15 | real signal | proceed |
| B2 ≈ B3 | all signal is in sector & category | redesign |

In [ ]:
# Environment and global configuration

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
DATA_PATH = Path("../01 EDA + Data PreProcessing/04_Five_CSV/03_financial_change_labels.csv")
OUT_DIR = Path("baseline_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The 9 change metrics
METRICS = [
    "current_assets", "fixed_assets", "creditors_total", "equity",
    "net_assets_liabilities", "cash", "debtors", "employees",
    "total_assets_less_current_liabilities",
]

# Metadata visible at time t
META_FEATURES = ["primary_sector", "Accounts_AccountCategory", "evidence_tier_t"]

# Experiment parameters
N_SPLITS = 5              # number of CV folds
TIME_HOLDOUT_FRAC = 0.20  # time holdout fraction
MIN_GROUP_N = 30          # min group size for B2
MIN_ROWS = 200            # skip metrics with too few rows
RANDOM_STATE = 0

## 1. Load data and sanity check

- `CompanyNumber_norm` Must be read as string, otherwise leading zeros are lost.
- The grain is a pair, not a company. One company may contribute several rows, which is also why the split is grouped by company.

In [2]:
# Load
df = pd.read_csv(DATA_PATH, dtype={"CompanyNumber_norm": str}, low_memory=False)

for col in ["period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

n_rows, n_companies = len(df), df["CompanyNumber_norm"].nunique()
print(f"rows: {n_rows:,}")
print(f"companies: {n_companies:,}")
print(f"rows/companies : {n_rows / n_companies:.2f}")

# Eligibility coverage per metric
cov = pd.DataFrame({
    "metric": METRICS,
    "eligible_n": [int(df[f"{m}_change_eligible"].fillna(False).sum()) for m in METRICS],
})
cov["eligible_pct"] = (cov["eligible_n"] / n_rows).round(3)
cov["zero_change_pct"] = [
    round(float((df[f"{m}_signed_log_change"] == 0).sum()
                / max(df[f"{m}_change_eligible"].fillna(False).sum(), 1)), 3)
    for m in METRICS
]
print("\nCoverage and zero-change share")
print(cov.to_string(index=False))

rows: 81,603
companies: 76,611
rows/companies : 1.07

Coverage and zero-change share
                               metric  eligible_n  eligible_pct  zero_change_pct
                       current_assets       69809         0.855            0.049
                         fixed_assets       40192         0.493            0.240
                      creditors_total       74018         0.907            0.065
                               equity       78768         0.965            0.127
               net_assets_liabilities       67001         0.821            0.061
                                 cash       31753         0.389            0.041
                              debtors       26618         0.326            0.112
                            employees       76322         0.935            0.734
total_assets_less_current_liabilities       70435         0.863            0.062


## 2. Feature construction (simplified)

Use only information visible at `available_date_t`.

Included：
- signed-log of the nine `_t` raw values & missing indicators
- `primary_sector`、`Accounts_AccountCategory`、`evidence_tier_t`
- month of `period_t`
- `gap_days`

excluded：
- Year or any linear time index.
- Any `_t_plus_1` / `_change_` / `_band` / `_threshold` column (all are t+1 information).

The next cell contains a hard assertion enforcing this whitelist.

In [3]:
def signed_log1p(x):
    """Sign-preserving log transform: sign(x)*ln(1+|x|)"""
    x = pd.to_numeric(x, errors="coerce")
    return np.sign(x) * np.log1p(np.abs(x))


# blacklist
FORBIDDEN_PATTERNS = ["_t_plus_1", "_change_", "_band", "_threshold", "percent_change"]


def build_features(frame, add_leak_column=None):
    """
    Build the time-t feature matrix.

    add_leak_column: Sanity check only — deliberately injects t+1 information.
    Returns: (X, categorical_column_indices)
    """
    X = pd.DataFrame(index=frame.index)

    # 9 t-period values & missing indicators
    for m in METRICS:
        X[f"{m}_t_sl"] = signed_log1p(frame[f"{m}_t"])
        X[f"{m}_t_missing"] = frame[f"{m}_t"].isna().astype(int)

    # time: month only, no year
    X["period_month"] = frame["period_t"].dt.month
    X["gap_days"] = frame["gap_days"]

    # categorical variables (-1 means missing)
    cat_idx = []
    for c in META_FEATURES:
        X[c] = frame[c].astype("category").cat.codes
        cat_idx.append(X.columns.get_loc(c))

    if add_leak_column is not None:
        X["__LEAK__"] = signed_log1p(frame[f"{add_leak_column}_t_plus_1"])

    # hard assertion
    bad = [c for c in X.columns if c != "__LEAK__"
           and any(p in c for p in FORBIDDEN_PATTERNS)]
    assert not bad, f"leaking features detected: {bad}"

    return X, cat_idx

# check
_X, _cat = build_features(df.head(100))
print(f"n_features : {_X.shape[1]}")
print(f"categorical: {[_X.columns[i] for i in _cat]}")

n_features : 23
categorical: ['primary_sector', 'Accounts_AccountCategory', 'evidence_tier_t']


## 3. Four baselines

|  | Prediction | Meaning |
|---|---|---|
| **B0** | constant 0 | Assume no change. Since `sl(t+1) − sl(t) = 0` means value unchanged, this is the naive persistence forecast — the bar the model must clear. |
| **B1** | train-fold mean of y | Assume everyone moves at the sample-average rate. The B0/B1 gap reveals overall drift. |
| **B2** | sector × category group mean | Tests how much sector and account category alone explain. |
| **B3** | HistGradientBoostingRegressor, default params | the question is whether signal exists, not how much can be squeezed out. |

B2's group means must be fitted inside the training fold only. Fitting on the full data before CV inflates this baseline and inverts the conclusion.

In [4]:
def fit_group_mean(y_train, keys_train, min_n=MIN_GROUP_N):
    """
    B2: Fit sector * category means inside the training fold; small groups fall back to global.
    """
    tmp = pd.DataFrame({"key": keys_train.values, "y": y_train.values})
    stats = tmp.groupby("key")["y"].agg(["mean", "count"])
    lookup = stats.loc[stats["count"] >= min_n, "mean"].to_dict()
    return lookup, float(y_train.mean())


def apply_group_mean(lookup, global_mean, keys):
    """Apply the fitted group means."""
    return keys.map(lookup).fillna(global_mean).to_numpy()


def make_group_key(frame):
    """Grouping key for B2."""
    return (frame["primary_sector"].astype(str) + "||"
            + frame["Accounts_AccountCategory"].astype(str))

## 4. Metrics

**Not $R^2$ alone.**

| Metric | Definition | Why |
|---|---|---|
| `skill_vs_zero` | `1 − MSE(model) / MSE(B0)` | Primary. Benchmarked explicitly against "no change". |
| `r2` | `1 − MSE / Var(y_test)` | Convention only; denominator uses the test set's own mean. |
| `spearman` | rank correlation | **The one that matters most**: robust to tails, measures ranking ability. |
| `decile_spread` | The difference between the true mean of change of the highest decile of the predicted value and the lowest decile | The most business-legible metric: is this lead list usable? |
| `mae` & `median_ae` | absolute error | If MAE is fine but skill is poor, the tails dominate. |
| `direction_acc` | The symbol consistency rate after excluding the no-change line | Compare against `majority_direction`. |

**importtant** alongside: ~74% of `employees` and ~25% of`fixed_assets` rows have exactly zero change, which determines how strong B0 is.


In [5]:
def evaluate(y_true, y_pred):
    """
    Compute the full metric set.
    y_true, y_pred: 1-D array-like
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mse = np.mean((y_true - y_pred) ** 2)
    mse_zero = np.mean(y_true ** 2)                    # MSE of B0
    mse_mean = np.mean((y_true - y_true.mean()) ** 2)  # R^2 denominator

    out = {
        "skill_vs_zero": 1 - mse / mse_zero if mse_zero > 0 else np.nan,
        "r2": 1 - mse / mse_mean if mse_mean > 0 else np.nan,
        "mae": float(np.mean(np.abs(y_true - y_pred))),
        "median_ae": float(np.median(np.abs(y_true - y_pred))),
    }

    # rank correlation
    out["spearman"] = (float(spearmanr(y_true, y_pred).statistic)
                       if np.std(y_pred) > 1e-12 else np.nan)

    # direction accuracy, excluding exact zeros
    nz = y_true != 0
    if nz.sum():
        out["direction_acc"] = float(np.mean(np.sign(y_pred[nz]) == np.sign(y_true[nz])))
        out["majority_direction"] = float(max(np.mean(y_true[nz] > 0),
                                              np.mean(y_true[nz] < 0)))
    else:
        out["direction_acc"] = out["majority_direction"] = np.nan

    # decile lift
    if np.std(y_pred) > 1e-12 and len(y_true) >= 100:
        rank = pd.qcut(pd.Series(y_pred).rank(method="first"), 10, labels=False)
        top, bottom = y_true[rank == 9].mean(), y_true[rank == 0].mean()
        out.update(decile_top_mean=float(top),
                   decile_bottom_mean=float(bottom),
                   decile_spread=float(top - bottom))
    else:
        out.update(decile_top_mean=np.nan, decile_bottom_mean=np.nan,
                   decile_spread=np.nan)

    return out

## 5. Splitting strategy

**5-fold GroupKFold by company**

81,603 pairs come from 76,611 companies; one company can contribute several rows. A random row-wise split would place two pairs of the same company on both sides.

**Secondary: time holdout by `available_date_t_plus_1`**

Deployment extrapolates forward, so a random split flatters the model. The time holdout takes the last 20% and additionally drops companies seen in training.

**test both** A obviously worse time holdout indicates distribution drift.

In [6]:
def run_metric(frame, metric, mode="group_cv", shuffle_y=False, leak_check=False):
    """
    Run one full evaluation for a single metric.

    mode : "group_cv" | "time_holdout"
    shuffle_y: label-shuffle sanity check
    leak_check: leak sanity check
    Returns: list of dict, one record per fold per model
    """
    # keep rows eligible for THIS metric
    elig = frame[f"{metric}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig].copy()
    y = sub[f"{metric}_signed_log_change"].astype(float)
    sub, y = sub.loc[y.notna()], y.loc[y.notna()]
    if len(sub) < MIN_ROWS:
        return []

    X, cat_idx = build_features(sub, add_leak_column=metric if leak_check else None)
    groups = sub["CompanyNumber_norm"]
    keys = make_group_key(sub)

    if shuffle_y:  # expect skill ≈ 0
        rng = np.random.default_rng(RANDOM_STATE)
        y = pd.Series(rng.permutation(y.to_numpy()), index=y.index)

    # splitting
    if mode == "group_cv":
        splits = list(GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups))
    elif mode == "time_holdout":
        cutoff = sub["available_date_t_plus_1"].quantile(1 - TIME_HOLDOUT_FRAC)
        is_test = (sub["available_date_t_plus_1"] > cutoff).to_numpy().copy()
        # drop companies also seen in training
        is_test &= ~groups.isin(groups[~is_test]).to_numpy()
        if is_test.sum() < 100:
            return []
        splits = [(np.where(~is_test)[0], np.where(is_test)[0])]
    else:
        raise ValueError(f"unknown mode: {mode}")

    records = []
    for fold, (tr, te) in enumerate(splits):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        preds = {
            "B0_zero": np.zeros(len(y_te)),
            "B1_train_mean": np.full(len(y_te), y_tr.mean()),
        }

        lookup, gmean = fit_group_mean(y_tr, keys.iloc[tr])   # train fold only
        preds["B2_group_mean"] = apply_group_mean(lookup, gmean, keys.iloc[te])

        model = HistGradientBoostingRegressor(
            max_iter=300, categorical_features=cat_idx, random_state=RANDOM_STATE
        )
        model.fit(X_tr, y_tr)
        preds["B3_gbm"] = model.predict(X_te)

        for name, pred in preds.items():
            records.append({
                "metric": metric, "mode": mode, "fold": fold, "model": name,
                "n_train": len(tr), "n_test": len(te),
                "zero_change_frac": float((y_te == 0).mean()),
                "y_std": float(y_te.std()),
                **evaluate(y_te, pred),
            })
    return records

## 6. Run the main experiment

Nine metrics × two splitting modes × four baselines.

In [8]:
all_records = []
for metric in METRICS:
    for mode in ("group_cv", "time_holdout"):
        all_records += run_metric(df, metric, mode=mode)
    print(f"  finished: {metric}")

results = pd.DataFrame(all_records)
results.to_csv(OUT_DIR / "baseline_folds.csv", index=False, encoding="utf-8-sig")
print(f"\n{len(results)} fold-level records")

  finished: current_assets
  finished: fixed_assets
  finished: creditors_total
  finished: equity
  finished: net_assets_liabilities
  finished: cash
  finished: debtors
  finished: employees
  finished: total_assets_less_current_liabilities

216 fold-level records


In [11]:
def summarise(res, mode):
    """Average across folds, keep SD for stability."""
    return (res[res["mode"] == mode]
            .groupby(["metric", "model"])
            .agg(n_test=("n_test", "sum"),
                 zero_frac=("zero_change_frac", "mean"),
                 skill=("skill_vs_zero", "mean"),
                 skill_sd=("skill_vs_zero", "std"),
                 r2=("r2", "mean"),
                 spearman=("spearman", "mean"),
                 mae=("mae", "mean"),
                 decile_spread=("decile_spread", "mean"))
            .round(4).reset_index())


summary_cv = summarise(results, "group_cv")
summary_time = summarise(results, "time_holdout")

summary_cv.to_csv(OUT_DIR / "summary_group_cv.csv", index=False, encoding="utf-8-sig")
summary_time.to_csv(OUT_DIR / "summary_time_holdout.csv", index=False, encoding="utf-8-sig")

print("GroupKFold")
print(summary_cv.to_string(index=False))
print("\nTime holdout")
print(summary_time.to_string(index=False))

GroupKFold
                               metric         model  n_test  zero_frac   skill  skill_sd      r2  spearman    mae  decile_spread
                                 cash       B0_zero   31753     0.0406  0.0000    0.0000 -0.0000       NaN 0.9701            NaN
                                 cash B1_train_mean   31753     0.0406 -0.0000    0.0000 -0.0001       NaN 0.9702            NaN
                                 cash B2_group_mean   31753     0.0406 -0.0002    0.0011 -0.0002    0.0281 0.9736         0.2561
                                 cash        B3_gbm   31753     0.0406  0.1492    0.0108  0.1491    0.2463 0.9807         2.1563
                      creditors_total       B0_zero   74018     0.0652  0.0000    0.0000 -0.0022       NaN 0.7027            NaN
                      creditors_total B1_train_mean   74018     0.0652  0.0021    0.0009 -0.0001       NaN 0.7071            NaN
                      creditors_total B2_group_mean   74018     0.0652  0.0021    0.00

## 7. Sanity checks

**A. Shuffled labels**
Expect skill ≈ 0 and Spearman ≈ 0. A materially non-zero result means the split or the features leak.

> It should return to near zero.

**B. Deliberate leak**
Inject `<metric>_t_plus_1` as a feature; expect a high skill score. If not, the
pipeline itself is broken (alignment, indexing, filtering).

> Anything above 0.8 counts as passing.

In [9]:
probe_metric = "current_assets"   # or any well-covered metric

print(f"probe metric: {probe_metric}\n")
for label, kwargs, expectation in [
    ("shuffled", dict(shuffle_y=True), "expect ≈ 0"),
    ("leaked",  dict(leak_check=True), "expect > 0.8"),
]:
    recs = run_metric(df, probe_metric, mode="group_cv", **kwargs)
    gbm = [r for r in recs if r["model"] == "B3_gbm"]
    if gbm:
        print(f"  {label:18s} skill={np.mean([r['skill_vs_zero'] for r in gbm]):+.3f}  "
              f"spearman={np.mean([r['spearman'] for r in gbm]):+.3f}   ({expectation})")

probe metric: current_assets

  shuffled           skill=-0.001  spearman=-0.005   (expect ≈ 0)
  leaked             skill=+0.997  spearman=+0.994   (expect > 0.8)


## 8. Diagnostic: is the signal just mean reversion?

The target is `sl(t+1) − sl(t)` while `sl(t)` is itself a feature, so **the target contains the negative of one feature by construction**. If levels mean-revert, a model can score well merely by learning "high `sl(t)` => negative change", which says nothing about expansion or financing need.

**Baseline B2c**: the metric's own t-period level as the sole feature.

| Outcome | Meaning | Next step |
|---|---|---|
| B2c ≈ B3 | all mean reversion |  |
| B2c << B3 | cross-metric structure matters | proceed |

In [15]:
def run_univariate(frame, metric):
    """
    B2c the metric's own t-period level only.
    Uses exactly the same filtering and splitting as run_metric.
    """
    elig = frame[f"{metric}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig].copy()
    y = sub[f"{metric}_signed_log_change"].astype(float)
    sub, y = sub.loc[y.notna()], y.loc[y.notna()]
    if len(sub) < MIN_ROWS:
        return []

    X = pd.DataFrame({f"{metric}_t_sl": signed_log1p(sub[f"{metric}_t"])})
    groups = sub["CompanyNumber_norm"]

    records = []
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups)):
        model = HistGradientBoostingRegressor(max_iter=300, random_state=RANDOM_STATE)
        model.fit(X.iloc[tr], y.iloc[tr])
        records.append({
            "metric": metric, "mode": "group_cv", "fold": fold, "model": "B2c_own_level",
            "n_train": len(tr), "n_test": len(te),
            "zero_change_frac": float((y.iloc[te] == 0).mean()),
            "y_std": float(y.iloc[te].std()),
            **evaluate(y.iloc[te], model.predict(X.iloc[te])),
        })
    return records


In [17]:
# Run and merge into the main results table
uni_records = []
for metric in METRICS:
    uni_records += run_univariate(df, metric)
    print(f"  finished: {metric}")

results = pd.concat([results, pd.DataFrame(uni_records)], ignore_index=True)
results.to_csv(OUT_DIR / "baseline_folds.csv", index=False, encoding="utf-8-sig")

summary_cv = summarise(results, "group_cv")
summary_cv.to_csv(OUT_DIR / "summary_group_cv.csv", index=False, encoding="utf-8-sig")

# Side-by-side comparison of the three key baselines
pivot = (summary_cv[summary_cv["model"].isin(["B2_group_mean", "B2c_own_level", "B3_gbm"])]
         .pivot(index="metric", columns="model", values="skill")
         .rename(columns={"B2_group_mean": "skill_B2_group",
                          "B2c_own_level": "skill_B2c_own",
                          "B3_gbm": "skill_B3"}))
pivot["B3_minus_B2c"] = (pivot["skill_B3"] - pivot["skill_B2c_own"]).round(4)
pivot["B2c_share_of_B3"] = (pivot["skill_B2c_own"] / pivot["skill_B3"]).round(3)

print("B2c_share_of_B3 close to 1 means pure mean reversion")
print(pivot.round(4).to_string())

  finished: current_assets
  finished: fixed_assets
  finished: creditors_total
  finished: equity
  finished: net_assets_liabilities
  finished: cash
  finished: debtors
  finished: employees
  finished: total_assets_less_current_liabilities
B2c_share_of_B3 close to 1 means pure mean reversion
model                                  skill_B2_group  skill_B2c_own  skill_B3  B3_minus_B2c  B2c_share_of_B3
metric                                                                                                       
cash                                          -0.0002         0.1176    0.1492        0.0316            0.788
creditors_total                                0.0021         0.0930    0.1600        0.0670            0.581
current_assets                                 0.0007         0.0846    0.1008        0.0162            0.839
debtors                                       -0.0003         0.0789    0.1249        0.0460            0.632
employees                                   

## 9. Reading the results

Apply the decision rule in Cell 1.


1. Start with `zero_change_frac`, it determines how strong B0 is.
2. Then compare **B3 against B2**. If they are close, all the signal lives in sector and account category which means the model is only doing an industry profile.
3. Finally read **Spearman and `decile_spread`**. They may disagree with skill: skill says whether the model beats "do nothing", Spearman says whether it can rank.

In [13]:
# the verdict per metric
verdict_rows = []
for metric in METRICS:
    sub = summary_cv[summary_cv["metric"] == metric]
    if sub.empty:
        continue
    gbm = sub[sub["model"] == "B3_gbm"].iloc[0]
    grp = sub[sub["model"] == "B2_group_mean"].iloc[0]

    skill, rho = gbm["skill"], gbm["spearman"]
    if pd.notna(grp["skill"]) and abs(gbm["skill"] - grp["skill"]) < 0.01:
        verdict = "B2≈B3: signal is all in group vars"
    elif skill > 0.05 or (pd.notna(rho) and rho > 0.15):
        verdict = "real signal, proceed"
    elif pd.notna(rho) and rho >= 0.05:
        verdict = "ranking-only"
    else:
        verdict = "no signal"

    verdict_rows.append({
        "metric": metric, "n": int(gbm["n_test"]), "zero_frac": gbm["zero_frac"],
        "skill_B3": skill, "skill_B2": grp["skill"], "spearman_B3": rho,
        "decile_spread": gbm["decile_spread"], "verdict": verdict,
    })

verdicts = pd.DataFrame(verdict_rows)
verdicts.to_csv(OUT_DIR / "verdicts.csv", index=False, encoding="utf-8-sig")
print(verdicts.to_string(index=False))

                               metric     n  zero_frac  skill_B3  skill_B2  spearman_B3  decile_spread              verdict
                       current_assets 69809     0.0490    0.1008    0.0007       0.1653         1.3415 real signal, proceed
                         fixed_assets 40192     0.2399    0.0543    0.0006       0.1606         0.7733 real signal, proceed
                      creditors_total 74018     0.0652    0.1600    0.0021       0.2212         1.7719 real signal, proceed
                               equity 78768     0.1274    0.1409    0.0005       0.0569         6.6877 real signal, proceed
               net_assets_liabilities 67001     0.0608    0.1327    0.0006       0.0419         6.6932 real signal, proceed
                                 cash 31753     0.0406    0.1492   -0.0002       0.2463         2.1563 real signal, proceed
                              debtors 26618     0.1121    0.1249   -0.0003       0.1986         1.5770 real signal, proceed
        